### Agentic Rag with LangGraph

In [ ]:
import os
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.documents import Document

ModuleNotFoundError: No module named 'langchain_core.text_splitter'

In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
os.environ["OPEN_AI_KEY"] = os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(model="gpt-4.1", temperature=0)
embeddings = OpenAIEmbeddings()

TypeError: str expected, not NoneType

In [8]:
class AgentState(TypedDict):
    question: str
    documents: List[Document]
    answer: str
    needs_retrieval: bool

In [ ]:
sample_texts =[

]

document = [Document(page_content=text) for text in sample_texts]

#creating vector store
vectorstore = FAISS.from_documents(documents,embeddings)
retriever = vectorstore.as_retriever(k=3)

In [ ]:
###Agentic functions

def decide_retrieval(state: AgentState)-> AgentState:
    """
        decide if we need to retreive document based on question
    """

    question= state['question']

    retrieval_keywords = ["what","how","where","explain","tell me"]
    needs_retrieval = any(keyword in question.lower() for keyword in retrieval_keywords)

    return {**state, "needs_retrieval": needs_retrieval}

In [ ]:
def retieve_documents(state: AgentState)->AgentState:
    question= state["question"]
    documents=retriever.invoke(question)

    return {**state, "documents": documents}

In [ ]:
def generate_answer(state:AgentState)->AgentState:
    question = state["question"]
    documents = state.get("docuemnts",[])

    if documents:
        context = "\n\n".join([doc.page_content for doc in documents])
        prompt=f"""Based on following context , answer the question:

        context:{context}
        Question:{question}
        Answer:"""
    else:
        prompt=f"""Based on the information available, I cannot answer the question.

        Question:{question}
        Answer:"""
    answer = llm.invoke(prompt)
    state["answer"] = answer
    return state

